# SafeSound 기준선 학습 (v1)

**셀을 위에서부터 순서대로 실행한다.** 셀 2는 런타임을 재시작시키므로, 재시작 후
셀 3부터 이어서 누르면 된다.

- 데이터: `dataset-v1` 태그 (창 13,007 / 5,207, int8 1.2초 저장)
- 모델: KWS20 v3 파생 1D CNN, 5클래스, QAT 8bit
- 산출물 1순위는 **혼동행렬**이다 — 배경음이 이벤트 합의 2.8배라 전체 정확도는
  전부 배경음으로 찍어도 74%가 나온다

> 런타임 유형을 **GPU (T4)** 로 먼저 바꿀 것. 수정 → 노트 설정 → 하드웨어 가속기.

## 0. Drive 에 올려둬야 하는 파일

| 파일 | 내용 |
|---|---|
| `MyDrive/max78000/safesound-v1.tar.gz` + `.sha256` | 데이터셋 (140MB) |
| `MyDrive/max78000/safesound-code-v1.tar.gz` + `.sha256` | 우리 코드 (수백 KB — 크기는 커밋마다 달라진다) |
| `MyDrive/max78000/ai8x-conda-env.tar.gz` *(선택)* | 환경 백업 3.2GB — 있으면 셀 3이 15분 아낀다 |

없으면 셀 3이 환경을 처음부터 만든다.

In [ ]:
# 1. Drive 마운트 — 데이터·코드·체크포인트가 모두 여기에 있다
import os

from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/max78000'
need = ['safesound-v1.tar.gz', 'safesound-v1.tar.gz.sha256',
        'safesound-code-v1.tar.gz', 'safesound-code-v1.tar.gz.sha256']
for f in need:
    print(('있음    ' if os.path.exists(f'{BASE}/{f}') else '없음 ✗  ') + f)
print(('있음    ' if os.path.exists(f'{BASE}/ai8x-conda-env.tar.gz')
       else '없음(새로 빌드)  ') + 'ai8x-conda-env.tar.gz')
missing = [f for f in need if not os.path.exists(f'{BASE}/{f}')]
assert not missing, f'Drive 에 없다: {missing}'

In [ ]:
# 2. condacolab 설치 — ai8x 는 Python 3.11 을 요구하는데 Colab 기본은 3.13 이다 (런타임 재시작됨)
!pip install -q condacolab
import condacolab

condacolab.install()

In [ ]:
# 3. ai8x 환경 준비 — Drive 백업이 있으면 복원, 없으면 CLAUDE.md 12장 해결책을 적용해 새로 만든다
import os
import subprocess

BASE = '/content/drive/MyDrive/max78000'
ENV = '/usr/local/envs/ai8x'


def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise SystemExit(f'실패: {cmd}')
    return r.returncode


# 어느 경로든 필요하다: torchaudio 의 speed_augment 가 시스템 libsox 를 쓴다.
# conda 환경 밖이라 환경 백업에 포함되지 않는다 (CLAUDE.md 12장)
sh('apt-get -qq install -y libsox3 libsox-dev libsox-fmt-all sox')
# Colab 의 matplotlib_inline 백엔드가 conda 환경에 없다
os.environ['MPLBACKEND'] = 'Agg'

if os.path.exists(ENV):
    print('환경이 이미 있다')
elif os.path.exists(f'{BASE}/ai8x-conda-env.tar.gz'):
    print('Drive 백업에서 복원한다 (약 3분)')
    sh(f'mkdir -p {ENV}')
    sh(f'tar xzf {BASE}/ai8x-conda-env.tar.gz -C /usr/local/envs')
else:
    print('환경을 새로 만든다 (약 15~20분)')
    sh('conda create -y -n ai8x python=3.11')
    # torch 2.3.1 은 PyPI 에서 내려갔다 → PyTorch 공식 cu121 인덱스에서 받는다
    sh('conda run -n ai8x --no-capture-output pip install -q '
       'torch==2.3.1 torchvision==0.18.1 '
       '--index-url https://download.pytorch.org/whl/cu121')

print()
sh('conda run -n ai8x --no-capture-output python -c '
   '"import sys, torch; print(sys.version.split()[0], torch.__version__, '
   'torch.cuda.is_available())"')

In [ ]:
# 4. ai8x-training 을 받고 CLAUDE.md 12장의 패치를 적용한다 (distiller 충돌·예외 처리)
import os
import subprocess


def sh(cmd, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True)
    if check and r.returncode:
        raise SystemExit(f'실패: {cmd}')
    return r.returncode


if not os.path.exists('/content/ai8x-training'):
    sh('git clone -q --recursive '
       'https://github.com/analogdevicesinc/ai8x-training.git /content/ai8x-training')

os.chdir('/content/ai8x-training')

# (1) distiller 디렉터리명이 네임스페이스 패키지 충돌을 만든다 —
#     /content/ai8x-training/distiller 가 설치된 패키지를 가린다
if os.path.isdir('distiller') and not os.path.isdir('distiller_repo'):
    sh('mv distiller distiller_repo')

# (2) distiller/__init__.py 의 except pkg_resources.DistributionNotFound 는
#     ContextualVersionConflict 를 못 잡아 모듈 로딩이 중간에 끊긴다.
#     knowledge_distillation 없음 / __file__ None / __version__ 없음 의 공통 원인
init = 'distiller_repo/distiller/__init__.py'
if os.path.exists(init):
    s = open(init, encoding='utf-8').read()
    if 'except pkg_resources.DistributionNotFound' in s:
        s = s.replace('except pkg_resources.DistributionNotFound', 'except Exception')
        open(init, 'w', encoding='utf-8').write(s)
        print('distiller __init__ 예외 처리 패치 적용')

# 환경 백업을 복원한 경우 이미 설치돼 있다. 실패해도 진행한다 (check=False)
sh('conda run -n ai8x --no-capture-output pip install -q -r requirements.txt', check=False)
sh('conda run -n ai8x --no-capture-output pip install -q -e distiller_repo', check=False)

# 로그·체크포인트는 Drive 에 둔다 — 세션이 끊겨도 남고, 재실행 시 여기서 재개한다
sh('rm -rf logs')
sh('mkdir -p /content/drive/MyDrive/max78000/logs')
sh('ln -s /content/drive/MyDrive/max78000/logs logs')
print()
print('logs ->', os.path.realpath('logs'))

In [ ]:
# 5. 우리 코드 tar 를 해시 확인 후 풀고, ai8x 쪽에 심볼릭 링크를 건다
import hashlib
import json
import os
import tarfile
import time

BASE = '/content/drive/MyDrive/max78000'


def verify(path, manifest=None):
    '''해시를 확인하고, 틀리면 **어느 파일이 옛것인지** 알려준다.

    예전에는 "불일치 ✗" 만 찍고 죽어서, tar 가 옛것인지 .sha256 만 옛것인지
    구별할 수 없었다. Drive 업로드는 둘 중 하나만 갱신되는 일이 흔하다
    (같은 이름을 덮어쓰지 않고 "사본"으로 추가하는 경우도 있다).
    '''
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for c in iter(lambda: f.read(1 << 20), b''):
            h.update(c)
    got = h.hexdigest()
    want = open(path + '.sha256').read().split()[0]
    name = os.path.basename(path)
    if got == want:
        print(f'{name}  OK  {got[:16]}...  {os.path.getsize(path):,} B')
        return
    print(f'{name}  불일치 ✗')
    print(f'  tar 실측   {got}')
    print(f'  사이드카   {want}')
    print(f'  tar 크기   {os.path.getsize(path):,} B')
    print(f'  tar mtime  {time.ctime(os.path.getmtime(path))}')
    print(f'  .sha256 mtime {time.ctime(os.path.getmtime(path + ".sha256"))}')
    if manifest:
        # tar 안의 커밋을 읽으면 어느 버전이 Drive 에 있는지 확정된다
        try:
            with tarfile.open(path) as tf:
                m = json.load(tf.extractfile(manifest))
            print(f'  tar 안의 커밋 {m["commit"][:9]} '
                  f'({len(m["files"])}개 경로, {m["created"]})')
        except Exception as e:
            print(f'  tar 를 열 수 없다 ({e}) → 업로드가 잘린 것이다')
    sib = sorted(f for f in os.listdir(os.path.dirname(path))
                 if f.startswith(os.path.splitext(name)[0][:14]))
    print(f'  같은 폴더의 비슷한 이름: {sib}')
    print('  판독: 실측이 기대값과 같은데 사이드카만 다르면 .sha256 만 다시')
    print('        올리면 된다. 실측·크기가 둘 다 다르면 tar 가 옛것이다.')
    print('        "사본" 파일이 보이면 덮어쓰기가 안 된 것이다.')
    raise AssertionError(f'{path} — tar 와 .sha256 이 서로 다른 버전이다')


verify(f'{BASE}/safesound-code-v1.tar.gz', manifest='CODE_MANIFEST.json')
os.makedirs('/content/max78000-proj', exist_ok=True)
with tarfile.open(f'{BASE}/safesound-code-v1.tar.gz') as tf:
    tf.extractall('/content/max78000-proj')

# 벤더 레포는 고치지 않고 링크만 건다 (CLAUDE.md 11장)
# melfeat.py / safesound_mel.py / ai85net-safesound-mel.py 는 G8 구성 ① 용이다.
for rel in ['datasets/safesound.py', 'models/ai85net-safesound.py',
            'datasets/melfeat.py', 'datasets/safesound_mel.py',
            'models/ai85net-safesound-mel.py']:
    dst = f'/content/ai8x-training/{rel}'
    if os.path.islink(dst) or os.path.exists(dst):
        os.remove(dst)
    os.symlink(f'/content/max78000-proj/{rel}', dst)
    print('link', dst, '->', os.readlink(dst))

print()
print('코드 번들 커밋:',
      json.load(open('/content/max78000-proj/CODE_MANIFEST.json'))['commit'][:9])

In [ ]:
# 6. 데이터셋 해시 확인 후 압축 해제 — 경로는 data/SafeSound/{train,test}/{클래스}/ 가 된다
import hashlib
import json
import os
import tarfile

BASE = '/content/drive/MyDrive/max78000'
DATA = '/content/ai8x-training/data/SafeSound'

want = open(f'{BASE}/safesound-v1.tar.gz.sha256').read().split()[0]
h = hashlib.sha256()
with open(f'{BASE}/safesound-v1.tar.gz', 'rb') as f:
    for c in iter(lambda: f.read(1 << 20), b''):
        h.update(c)
got = h.hexdigest()
if got != want:
    # 셀 5 와 같은 이유로 두 해시를 모두 찍는다 — tar 가 옛것인지 .sha256 만
    # 옛것인지 구별하려면 값이 필요하다
    print('데이터 tar 불일치 ✗')
    print('  tar 실측 ', got)
    print('  사이드카 ', want)
    print('  tar 크기 ', f'{os.path.getsize(f"{BASE}/safesound-v1.tar.gz"):,} B')
    raise AssertionError('데이터 tar 와 .sha256 이 서로 다른 버전이다')
print('sha256 OK', want[:16], '...')

os.makedirs(DATA, exist_ok=True)
with tarfile.open(f'{BASE}/safesound-v1.tar.gz') as tf:
    tf.extractall(DATA)

m = json.load(open(f'{DATA}/MANIFEST.json'))
print()
print('데이터셋', m['tag'], '/ 커밋', m['commit'][:9])
for k, v in sorted(m['counts'].items()):
    print(f'  {k:<22}{v:>7}창')
print(f'  {"합계":<22}{sum(m["counts"].values()):>7}창')

In [ ]:
# 7. 학습 전 필수 검증 — KAT(축·스케일)과 클래스 가중치. 여기서 실패하면 학습하지 말 것
import subprocess

CHECK = '''
import sys
sys.path.insert(0, "/content/ai8x-training")
sys.path.insert(0, "/content/max78000-proj/datasets")
import safesound as S
print("클래스 가중치(상수):", dict(zip(S.CLASSES, S.datasets[0]["weight"])))
print("실측 재계산       :",
      dict(zip(S.CLASSES, S.class_weights("/content/ai8x-training/data/SafeSound"))))
'''

# ⚠️ --data 를 반드시 넘긴다. 기본값은 WSL 경로라 Colab 에서는 샤드를 못 찾는데,
#    예전에는 그럴 때 "생략" 하고 통과로 보였다 (로더 검사가 한 번도 안 돌았다).
#    지금은 샤드를 못 찾으면 **실패**한다.
SHARDS = '/content/ai8x-training/data/SafeSound'
for kat in ('tools/kat_safesound.py', 'tools/kat_safesound_mel.py'):
    r = subprocess.run(f'cd /content/max78000-proj && conda run -n ai8x '
                       f'--no-capture-output python {kat} --data {SHARDS}',
                       shell=True)
    assert r.returncode == 0, f'{kat} 실패 — 전처리·로더가 어긋났다. 학습하지 말 것'

# 모델 forward 회귀 검사 — 마지막 배치(59샘플) + channels_last.
# 구성 ① 학습이 에폭 0 을 다 돌고 마지막 배치에서 view() 로 죽은 적이 있다.
# 몇 초 만에 잡히는 것을 몇 시간 뒤에 알게 되는 일을 막는다.
r = subprocess.run('cd /content/max78000-proj && conda run -n ai8x '
                   '--no-capture-output python tools/kat_models.py '
                   '--ai8x /content/ai8x-training', shell=True)
assert r.returncode == 0, 'kat_models 실패 — 모델 forward 가 깨진다. 학습하지 말 것'

open('/content/_check.py', 'w').write(CHECK)
subprocess.run('conda run -n ai8x --no-capture-output python /content/_check.py', shell=True)

In [ ]:
# 8. 기준선 학습 — 재실행하면 마지막 체크포인트에서 자동으로 이어서 간다 (진행 로그가 실시간으로 흐른다)
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'


def run_live(cmd):
    '''출력을 줄 단위로 흘려보낸다.

    subprocess.run(...) 은 Colab 에서 자식 출력이 버퍼링돼 학습이 끝날 때까지
    아무것도 안 보인다. 몇 시간짜리 작업에서 진행이 안 보이는 것은 곤란하다.
    '''
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        print(line, end='', flush=True)
        buf.append(line)
    p.wait()
    return p.returncode, ''.join(buf)

# distiller 는 logs/<name>___<타임스탬프>/ 를 만든다 → 가장 최근 것을 찾는다
ckpts = sorted(glob.glob('logs/safesound-v1___*/*checkpoint.pth.tar'),
               key=os.path.getmtime)
resume = f'--resume-from {ckpts[-1]}' if ckpts else ''
print('재개:', ckpts[-1] if ckpts else '(없음 — 처음부터)')

cmd = (
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    ' --deterministic --seed 1'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundnet --dataset SafeSound --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name safesound-v1 --out-dir logs {resume}'
)
print(cmd)
print()
rc, _ = run_live(cmd)
print()
print('종료 코드', rc, '— 0 이 아니면 위 로그의 마지막 오류를 볼 것')

In [ ]:
# 9. 평가 — 창/원본 혼동행렬, 원본 부트스트랩 CI, 시간당 오경보. 출력이 흐르면서 Drive 에도 저장된다
import glob
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'


def run_live(cmd):
    '''셀 8과 같은 이유로 줄 단위로 흘려보낸다 (부트스트랩 2,000회가 시간을 쓴다).'''
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        print(line, end='', flush=True)
        buf.append(line)
    p.wait()
    return p.returncode, ''.join(buf)


best = sorted(glob.glob('logs/safesound-v1___*/*best*.pth.tar'), key=os.path.getmtime)
assert best, 'best 체크포인트가 없다 — 셀 8을 먼저 끝낼 것'
print('평가 대상:', best[-1])
print()

# `-n ai8x` 가 빠지면 base 환경에서 돌아 ModuleNotFoundError 가 난다
cmd = (
    'conda run -n ai8x --no-capture-output python '
    '/content/max78000-proj/tools/eval_confusion.py '
    f'--checkpoint {best[-1]} --data /content/ai8x-training/data '
    '--ai8x /content/ai8x-training --hop-ms 250'
)
rc, text = run_live(cmd)

out = f'{BASE}/eval-{time.strftime("%Y%m%d-%H%M%S")}.txt'
open(out, 'w', encoding='utf-8').write(text)
print()
print('종료 코드', rc, '| 저장:', out)

## 10. 기준선 결과 분석 (셀 12~14)

첫 기준선에서 배경음 오탐 34%(시간당 4,896회)가 나왔다. **재학습 전에** 이미 있는
체크포인트로 세 가지를 확인한다.

1. 배경음 로짓 오프셋을 쓸어 오경보-재현율 곡선을 본다 → 운용 임계값 문제인지
   모델 문제인지 갈린다. **1차 결과: 곡선이 매우 완만했다** (b=−2~+8 전 구간에서
   7,279→2,623회/h, 100/10/1 전부 도달 불가). 그래서 셀 12 는 등간격 b 스윕 대신
   **마진 분위수**에서 문턱값을 집고, 남은 오경보를 **배경음 출처별로 분해**한다.
   마지막 표는 **조용한 창 쿼터 쏠림**이다 — "조용함 → siren" 을 배웠는지 본다
   (가설 근거는 `docs/results/input-stats-v1.md`, 대응은 `TASKS.md` v2 항목)
2. float `best` 와 `qat_best` 를 비교한다 → 손실이 QAT 때문인지 본다
3. 학습 곡선에서 QAT 전환 전후를 본다 → 전환이 회복됐는지 본다

In [ ]:
# 12. 오경보 문턱값 — 마진 분위수 스윕 + 배경음 출처별 분해 (+ 등간격 b 스윕)
import glob
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    buf = []
    for line in p.stdout:
        print(line, end='', flush=True)
        buf.append(line)
    p.wait()
    return p.returncode, ''.join(buf)


qat = sorted(glob.glob('logs/safesound-v1___*/*qat_best*.pth.tar'), key=os.path.getmtime)
assert qat, 'qat_best 체크포인트가 없다'
stamp = time.strftime('%Y%m%d-%H%M%S')
rc, text = run_live(
    'conda run -n ai8x --no-capture-output python '
    '/content/max78000-proj/tools/eval_threshold.py '
    f'--checkpoint {qat[-1]} --data /content/ai8x-training/data '
    '--ai8x /content/ai8x-training --hop-ms 250 '
    # b 는 테스트셋을 **원본 단위**로 반분해 한쪽에서 고르고 다른 쪽에 적용한다.
    # ai8x 검증셋(--pick-on val)은 창 단위 분리라 같은 녹음이 학습에 들어가 있어
    # 낙관적으로 나온다.
    '--pick-on test-half '
    # 마진 모드는 기본이다. 출처 태그는 샤드의 index.csv 에서 읽으므로 따로 올릴
    # 파일이 없다 (데이터셋별 분해만 매니페스트가 필요한데, 배경음이 전량
    # FSD50K 라 어차피 한 줄이다).
    f'--png {BASE}/threshold-sweep-{stamp}.png --json {BASE}/threshold-sweep-{stamp}.json')
open(f'{BASE}/threshold-{stamp}.txt', 'w', encoding='utf-8').write(text)
print()
print('종료 코드', rc)

In [ ]:
# 13. float best vs qat_best — 양자화로 얼마나 잃었는지 같은 지표로 비교한다
import glob
import json
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')


def newest(pat):
    f = sorted(glob.glob(pat), key=os.path.getmtime)
    return f[-1] if f else None


# qat_best 도 *best* 에 걸리므로 이름으로 갈라낸다
qat = newest('logs/safesound-v1___*/*qat_best*.pth.tar')
allbest = sorted(glob.glob('logs/safesound-v1___*/*best*.pth.tar'), key=os.path.getmtime)
flt = next((p for p in reversed(allbest) if 'qat' not in os.path.basename(p)), None)
print('float best:', flt)
print('qat   best:', qat)
assert flt and qat, 'float best 또는 qat_best 가 없다'

out = {}
for tag, ck in [('float', flt), ('qat', qat)]:
    j = f'{BASE}/eval-{tag}-{stamp}.json'
    subprocess.run(
        'conda run -n ai8x --no-capture-output python '
        '/content/max78000-proj/tools/eval_confusion.py '
        f'--checkpoint {ck} --data /content/ai8x-training/data '
        f'--ai8x /content/ai8x-training --hop-ms 250 --n-boot 500 --json {j}',
        shell=True)
    out[tag] = json.load(open(j))

names = list(out['float']['window']['recall'])
print()
print(f"{'클래스':<12}{'float 창':>10}{'qat 창':>10}{'차이':>9}   "
      f"{'float 원본':>11}{'qat 원본':>10}")
print('-' * 64)
for n in names:
    f_w = 100 * out['float']['window']['recall'][n]
    q_w = 100 * out['qat']['window']['recall'][n]
    f_o = 100 * out['float']['origin']['recall'][n]
    q_o = 100 * out['qat']['origin']['recall'][n]
    print(f'{n:<12}{f_w:>9.1f}%{q_w:>9.1f}%{q_w-f_w:>+8.1f}p   {f_o:>10.1f}%{q_o:>9.1f}%')
print()
for tag in ('float', 'qat'):
    s = out[tag]
    print(f"  {tag:<6} macro-F1 창 {s['window']['macro_f1']:.4f} / "
          f"원본 {s['origin']['macro_f1']:.4f} | "
          f"오경보 {s['false_alarm']['per_hour']:,.0f}회/h")

In [ ]:
# 14. 학습 곡선 — 에폭별 train/val Top1·loss 와 QAT 전환 전후 요약
import glob
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')

logs = sorted(glob.glob('logs/safesound-v1___*/*.log'), key=os.path.getmtime)
assert logs, '학습 로그가 없다'
print('로그:', logs[-1])
print()

p = subprocess.Popen(
    'conda run -n ai8x --no-capture-output python '
    '/content/max78000-proj/tools/parse_trainlog.py '
    f'--log "{logs[-1]}" --qat-epoch 60 --every 10 '
    f'--png {BASE}/curves-{stamp}.png',
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
buf = []
for line in p.stdout:
    print(line, end='', flush=True)
    buf.append(line)
p.wait()
open(f'{BASE}/curves-{stamp}.txt', 'w', encoding='utf-8').write(''.join(buf))

In [ ]:
# 15. 오분류 창을 wav 로 내보낸다 — 라벨 문제인지 모델 문제인지 귀로 가른다
#     (1) scream → siren/dog_bark 무작위 15개씩
#     (2) background → siren/dog_bark **확률 상위** 15개씩 (파일명에 출처 태그)
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'

qat = sorted(glob.glob('logs/safesound-v1___*/*qat_best*.pth.tar'), key=os.path.getmtime)
assert qat, 'qat_best 체크포인트가 없다'

JOBS = [
    # (정답, 예측, 표본 추출, 출처 태그, 저장 위치)
    ('scream', 'random', False, f'{BASE}/listen_scream_errors'),
    # 배경음은 **확률 상위**를 뽑는다. 모델이 확신한 오경보일수록 배경음 라벨에
    # 진짜 사이렌·개 짖음이 섞여 있을 확률이 높다 — 그게 이 청취의 쟁점이다.
    ('background', 'top', True, f'{BASE}/listen_bg_errors'),
]

for true_cls, pick, tag, out in JOBS:
    print(f'--- {true_cls} → siren / dog_bark  ({pick}) ---')
    cmd = ('conda run -n ai8x --no-capture-output python '
           '/content/max78000-proj/tools/export_errors.py '
           f'--checkpoint {qat[-1]} --data /content/ai8x-training/data '
           '--ai8x /content/ai8x-training '
           f'--true {true_cls} --pred siren dog_bark --n 15 '
           f'--pick {pick} --out {out}')
    if tag:
        cmd += ' --tag-source'
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print()
    print('Drive 에서 듣기:', out)
    print()

In [ ]:
# 16. 과소적합 판정 — train(무증강) vs test 클래스별 재현율을 나란히 본다
#
#     학습 로그의 train Top1 으로는 판정할 수 없다. 손실이 역빈도 가중(bg 0.27)
#     이라 모델이 배경음을 의도적으로 희생하는데 Top1 은 비가중이고, train Top1
#     은 증강이 걸린 데이터 기준이라 평가 조건이 아니다.
#     여기서는 **같은 체크포인트·같은 무증강 경로**로 두 split 을 돌려 비교한다.
#       · train 재현율도 낮다  → 학습 데이터조차 못 맞힌다 (용량·최적화·라벨)
#       · train 만 높다        → 일반화 문제 (데이터 양·다양성·증강)
import glob
import json
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')

qat = sorted(glob.glob('logs/safesound-v1___*/*qat_best*.pth.tar'), key=os.path.getmtime)
assert qat, 'qat_best 체크포인트가 없다'

out = {}
for split in ('train', 'test'):
    j = f'{BASE}/eval-{split}-{stamp}.json'
    subprocess.run(
        'conda run -n ai8x --no-capture-output python '
        '/content/max78000-proj/tools/eval_confusion.py '
        f'--checkpoint {qat[-1]} --data /content/ai8x-training/data '
        f'--ai8x /content/ai8x-training --split {split} '
        f'--hop-ms 250 --n-boot 500 --json {j}',
        shell=True)
    out[split] = json.load(open(j))

names = list(out['test']['window']['recall'])
print()
print(f"{'클래스':<12}{'train 창':>10}{'test 창':>10}{'차이':>9}   "
      f"{'train 원본':>11}{'test 원본':>11}")
print('-' * 65)
for n in names:
    tr = 100 * out['train']['window']['recall'][n]
    te = 100 * out['test']['window']['recall'][n]
    tro = 100 * out['train']['origin']['recall'][n]
    teo = 100 * out['test']['origin']['recall'][n]
    print(f'{n:<12}{tr:>9.1f}%{te:>9.1f}%{te-tr:>+8.1f}p   {tro:>10.1f}%{teo:>10.1f}%')
print()
for s in ('train', 'test'):
    d = out[s]
    print(f"  {s:<6} macro-F1 창 {d['window']['macro_f1']:.4f} / "
          f"원본 {d['origin']['macro_f1']:.4f} | "
          f"오경보 {d['false_alarm']['per_hour']:,.0f}회/h")
print()
print('  ⚠️ train split 에는 ai8x 가 떼어 간 검증 창(10%)도 섞여 있다.')
print('     과소적합 판정용 수치이지 일반화 성능이 아니다.')

## 17~19. G8 구성 ① — 로그 멜 + 2D CNN

구성 ④(raw 파형 1D CNN)의 대조군이다. **입력 표현만 다르고 나머지는 전부 같다** —
같은 샤드·같은 원본 단위 분할·같은 클래스 가중치·같은 증강·같은 스케줄·같은 QAT·
같은 평가 스크립트.

| | 구성 ④ (wave) | 구성 ① (mel) |
|---|---|---|
| 입력 | raw 파형 (128, 128) int8 | 로그 멜 (1, 64, 64) int8 |
| CPU 가 하는 일 | 없음 | STFT + 멜 + 로그 |
| params | 165,457 (36.6%) | 157,535 (34.8%) |

**①은 ④의 병목이 입력 표현인지 데이터인지 가리는 실험이다.** 기준선이 과소적합
쪽인지 아닌지와 무관하게 지금 돌린다 — ①도 같이 낮으면 데이터·라벨 문제이고,
①만 잘 나오면 raw 파형 표현이 병목이라는 뜻이다.

설계 근거: `docs/results/config1-mel2d-design.md`,
양자화 구간 근거: `docs/results/mel-range-sweep.md`.

In [ ]:
# 17. 구성 ① 특징 캐시 — 테스트셋은 무증강이라 한 번만 계산해 두면 평가가 빨라진다
#     (학습셋은 매 에폭 증강이 달라야 하므로 캐시하지 않는다 — 코드가 막아 둔다)
import os
import subprocess

BUILD = '''
import os, sys, time
sys.path.insert(0, "/content/ai8x-training")
sys.path.insert(0, "/content/max78000-proj/datasets")
import melfeat as MF, safesound_mel as SM
root = "/content/ai8x-training/data/SafeSound"
path = os.path.join(root, "test", f"melcache_{MF.N_MELS}x{MF.N_FRAMES}.npy")
if os.path.isfile(path):
    print("캐시 이미 있음:", path)
else:
    ds = SM.SafeSoundMel(root, "test", transform=None, augment=False)
    t0 = time.time()
    ds.build_cache(path)
    print(f"캐시 생성 {len(ds):,}창  {time.time()-t0:.0f}s  ->", path)
'''
open('/content/_melcache.py', 'w').write(BUILD)
subprocess.run('conda run -n ai8x --no-capture-output python /content/_melcache.py',
               shell=True)

In [ ]:
# 18. 구성 ① 학습 — ④와 **같은 스케줄·같은 QAT·같은 에폭**. 재실행하면 이어서 간다
import glob
import os
import subprocess

os.chdir('/content/ai8x-training')
os.environ['MPLBACKEND'] = 'Agg'
PROJ = '/content/max78000-proj'


def run_live(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    return p.returncode


ckpts = sorted(glob.glob('logs/safesound-mel-v1___*/*checkpoint.pth.tar'),
               key=os.path.getmtime)
resume = f'--resume-from {ckpts[-1]}' if ckpts else ''
print('재개:', ckpts[-1] if ckpts else '(없음 — 처음부터)')

# ⚠️ 아래 인자는 셀 8(구성 ④)과 **한 글자도 다르면 안 된다** — 모델·데이터셋·
#    이름만 빼고. 다르면 비교가 오염된다.
cmd = (
    'conda run -n ai8x --no-capture-output python train.py'
    ' --epochs 150 --optimizer Adam --lr 0.001 --wd 0'
    ' --deterministic --seed 1'
    f' --compress {PROJ}/colab/schedule_safesound.yaml'
    f' --qat-policy {PROJ}/colab/qat_policy_safesound.yaml'
    ' --model ai85safesoundmelnet --dataset SafeSoundMel --device MAX78000'
    ' --confusion --batch-size 128 --workers 2'
    f' --name safesound-mel-v1 --out-dir logs {resume}'
)
print(cmd)
print()
print('종료 코드', run_live(cmd))

In [ ]:
# 19. 구성 ① vs ④ — 같은 평가 스크립트, 같은 테스트셋. 이 표가 G8 의 최소 버전이다
import glob
import json
import os
import subprocess
import time

os.chdir('/content/ai8x-training')
BASE = '/content/drive/MyDrive/max78000'
stamp = time.strftime('%Y%m%d-%H%M%S')

RUNS = [('④ wave', 'wave', 'logs/safesound-v1___*/*qat_best*.pth.tar'),
        ('① mel', 'mel', 'logs/safesound-mel-v1___*/*qat_best*.pth.tar')]

out = {}
for label, cfg, pat in RUNS:
    ck = sorted(glob.glob(pat), key=os.path.getmtime)
    if not ck:
        print(f'{label}: 체크포인트 없음 — 건너뜀')
        continue
    j = f'{BASE}/eval-g8-{cfg}-{stamp}.json'
    subprocess.run(
        'conda run -n ai8x --no-capture-output python '
        '/content/max78000-proj/tools/eval_confusion.py '
        f'--checkpoint {ck[-1]} --data /content/ai8x-training/data '
        f'--ai8x /content/ai8x-training --config {cfg} '
        f'--hop-ms 250 --n-boot 2000 --json {j}', shell=True)
    out[label] = json.load(open(j))

if len(out) == 2:
    a, b = RUNS[0][0], RUNS[1][0]
    names = list(out[a]['window']['recall'])
    print()
    print(f"{'클래스':<12}{a+' 창':>12}{b+' 창':>12}{'차이':>9}   "
          f"{a+' 원본':>13}{b+' 원본':>13}")
    print('-' * 74)
    for n in names:
        aw = 100 * out[a]['window']['recall'][n]
        bw = 100 * out[b]['window']['recall'][n]
        ao = 100 * out[a]['origin']['recall'][n]
        bo = 100 * out[b]['origin']['recall'][n]
        print(f'{n:<12}{aw:>11.1f}%{bw:>11.1f}%{bw-aw:>+8.1f}p   '
              f'{ao:>12.1f}%{bo:>12.1f}%')
    print()
    for label in (a, b):
        d = out[label]
        print(f"  {label:<8} macro-F1 창 {d['window']['macro_f1']:.4f} / "
              f"원본 {d['origin']['macro_f1']:.4f} | "
              f"오경보 {d['false_alarm']['per_hour']:,.0f}회/h  (argmax)")

    # ── 고정 오경보 비교. **모델 비교의 기준은 이쪽이다.**
    #    argmax 는 각 모델이 알아서 정한 동작점이라, 배경음을 더 버린 모델이
    #    이벤트 재현율만 높게 보인다. 오경보를 같은 값으로 고정해야 공정하다.
    for pt in ('1000/h', '300/h'):
        rows = {}
        for label in (a, b):
            fx = (out[label].get('fixed_fa') or {}).get('targets', {})
            if fx.get(pt):
                rows[label] = fx[pt]
        if len(rows) < 2:
            print()
            print(f'  ({pt} 지점 없음 — 코드 번들이 오래된 것일 수 있다)')
            continue
        print()
        print(f'=== 고정 오경보 {pt} 에서의 클래스별 재현율 ===')
        print(f"{'클래스':<12}{a:>12}{b:>12}{'차이':>9}")
        print('-' * 45)
        for n in names:
            av = 100 * rows[a]['recall'][n]
            bv = 100 * rows[b]['recall'][n]
            print(f'{n:<12}{av:>11.1f}%{bv:>11.1f}%{bv-av:>+8.1f}p')
        for label in (a, b):
            r = rows[label]
            print(f"  {label:<8} macro-F1 {r['macro_f1']:.4f}  "
                  f"실제 오경보 {r['per_hour']:.1f}회/h  문턱값 {r['thr']:.2f}")
    # ── 출처별 자체 오탐률 ①/④ 나란히.
    #    청취(2026-09-27)에서 배경음 오경보의 두 축이 갈렸다:
    #      알람 → siren  (음높이 궤적 차이 → 스펙트로그램이 유리할 것으로 예상)
    #      말소리 → dog_bark (배경음에 음성이 없어서 → 입력 표현과 무관할 것)
    #    두 예상이 갈리는지가 곧 "전처리 위치가 어떤 혼동에 듣는가" 다.
    sa = out[a].get('bg_sources') or {}
    sb = out[b].get('bg_sources') or {}
    if sa and sb:
        print()
        print('=== 배경음 출처별 자체 오탐률 (argmax) ①/④ 비교 ===')
        print(f"{'출처':<26}{'창':>7}{a+' 전체':>12}{b+' 전체':>12}"
              f"{a+'→siren':>13}{b+'→siren':>13}{'siren 차이':>12}")
        print('-' * 96)
        for tag in sa:
            if tag not in sb:
                continue
            da, db = sa[tag], sb[tag]
            ra = 100 * da['by_class_rate']['siren']
            rb = 100 * db['by_class_rate']['siren']
            print(f"{tag:<26}{da['n']:>7,}{100*da['fa_rate']:>11.1f}%"
                  f"{100*db['fa_rate']:>11.1f}%{ra:>12.1f}%{rb:>12.1f}%"
                  f"{rb-ra:>+11.1f}p")
        print()
        print('  ★ 핵심은 `하드네거티브:alarm → siren` 행이다.')
        print('    사이렌은 음높이가 휘고 알람은 고정 음높이 반복이라,')
        print('    스펙트로그램에서는 기울어진 선 대 수평선이 된다 →')
        print('    3x3 conv 가 잡기 쉬우므로 ①에서 줄어들 것으로 예상한다.')
        print('    줄어들면 데이터를 건드릴 필요가 없고, 그 자체가 G8 의 결과다.')
        print('    줄지 않으면 알람을 별도 클래스로 세우는 것까지 검토한다.')
        print('  ☆ 대조: 말소리→dog_bark 는 배경음에 음성이 없어서 나는 것이라')
        print('    입력 표현과 무관할 것으로 본다 (v2 (b) LibriSpeech 추가 대상).')
        print('    두 예상이 실제로 갈리면 "전처리 위치가 어떤 혼동에 듣는가" 의')
        print('    구체 사례가 된다. 근거: docs/results/listening-verification.md 7.2')
    print()
    print('  ⚠️ 지연·에너지는 여기서 나오지 않는다 — 보드가 있어야 한다.')
    print('     이 표는 G8 의 **정확도 축**이고, 시간 축은 Phase 5 에서 채운다.')

## 20. 학습이 끝난 뒤 (WSL2 에서)

혼동행렬의 **N프레임 오경보는 하한(낙관치)** 이라 논문에 쓰지 않는다. 논문 수치는
원본 오디오를 빈틈없이 훑는 스트리밍 평가에서 나오고, 그건 원본이 있는 WSL2 에서
돌리는 편이 간단하다.

```bash
# WSL2 — best 체크포인트를 Drive 에서 내려받은 뒤
~/ai8x-training/venv/bin/python tools/stream_eval.py \
  --checkpoint <best.pth.tar> --ai8x ~/ai8x-training \
  --json docs/results/stream-eval-pc.json
```

그다음 양자화·합성으로 넘어간다 (442KB / 512KB 초과 여부 확인).

**읽는 순서**: 혼동행렬(원본 단위) → 클래스별 재현율·CI → 시간당 오경보.
전체 정확도는 보지 않는다.